# ML Lab · Unit 9: Colab + Gemini workshop

**Prof. Dr. Mehmet Songur** · ML Lab, Module 0

In this workshop you work with real data (US CPI, FRED: `CPIAUCNS`), have **Gemini write the code** and **audit it**.

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells (the Gemini button in a cell or the Gemini panel; the name may vary with Colab's interface).
3. When Gemini suggests code, first press **Accept** to put it in the cell; it does not run yet. Then **read it before running it**, audit it against the error catalogue from unit 6, fix it if needed, and only then run it. Are the checks you asked for really in the code, or did Gemini just print some numbers?
4. If a check cell passes, it gives you a **completion code**; paste it into the matching box in ML Lab.

> To keep your own copy: *File → Save a copy in Drive*.

In [ ]:
#@title Setup — run it, do not change it

import io, math, base64, urllib.request
import pandas as pd

def fred_download(series_id):
    """Downloads a series from FRED; columns: date, <series_id>. FRED marks missing values with '.';
    they are deliberately turned into NaN and counted."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + series_id
    with urllib.request.urlopen(url) as response:
        df = pd.read_csv(io.BytesIO(response.read()))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    print(f"{series_id}: {len(df)} rows, {df[series_id].isna().sum()} missing, {df['date'].min():%Y-%m} – {df['date'].max():%Y-%m}")
    return df

raw = fred_download("CPIAUCNS").rename(columns={"CPIAUCNS": "index"})

def _k(s):
    return base64.b64decode(s).decode()

def _true_annual():
    d = raw.sort_values("date").set_index("date")["index"]
    y = d.pct_change(periods=12, fill_method=None) * 100
    clean = d.isna().astype(int).rolling(13, min_periods=1).max() == 0   # windows containing a gap are not compared
    return y[clean]

def check_A(df):
    if "annual" not in df.columns:
        print("❌ No column called 'annual' was found."); return
    if len(df) != len(raw):
        print(f"❌ The row count changed: it should be {len(raw)}, it is {len(df)}. Which rows were dropped or duplicated?"); return
    if df["date"].duplicated().any():
        print("❌ Some dates appear more than once. Rows may have been duplicated."); return
    s = df.set_index("date")["annual"].sort_index()
    if not s.iloc[:12].isna().all():
        print("❌ In the date-sorted table the first 12 months should be empty (there is no value 12 months earlier). Did you sort by date before computing? Could the gaps have been filled?"); return
    true = _true_annual().dropna()
    common = s.reindex(true.index)
    if common.isna().any():
        print(f"❌ 'annual' is empty in {common.isna().sum()} computable months. There are unexpected missing values."); return
    if ((common * 100 - true).abs() < 1e-6).mean() > 0.95:
        print("❌ The values were computed as ratios. Did you convert to percent?"); return
    diff = (common - true).abs()
    if diff.max() > 1e-6:
        print(f"❌ {(diff > 1e-6).sum()} months differ from the expected values. Did you sort by date before computing? Did you use periods=12?"); return
    print("✅ All checks passed. Your completion code: " + _k("TUwwLUVORi03UUsy"))

def check_B(peak_month, peak_value):
    d = _true_annual()
    d = d[d.index >= "2000-01-01"].dropna()
    try:
        m = pd.Timestamp(peak_month)
    except Exception:
        print("❌ 'peak_month' cannot be converted to a date."); return
    try:
        value = float(peak_value)
    except Exception:
        print("❌ 'peak_value' is not a number."); return
    if not math.isfinite(value):
        print("❌ 'peak_value' is not a finite number (NaN or infinite). How were missing values handled?"); return
    if m.to_period("M") != d.idxmax().to_period("M"):
        print("❌ The peak month differs from the expected one. Did you exclude data before 2000? Did you use the annual series?"); return
    if abs(value - d.max()) > 0.05:
        print("❌ The peak value differs from the expected one. Is it in percent?"); return
    print("✅ All checks passed. Your completion code: " + _k("TUwwLVpSVi00TVRY"))

# Note: the rows are not in the order they were downloaded. Remember this when auditing.
cpi = raw.sample(frac=1, random_state=7).reset_index(drop=True)
print("cpi table:", cpi.shape)
cpi.head()

## Task A — Annual inflation

Add **annual inflation in percent** to the `cpi` table (columns: `date`, `index`; monthly) as a new column called `annual`.

**Sample prompt for Gemini (writing your own is better):**
> A pandas table called cpi has 'date' (monthly, datetime) and 'index' columns. Sort the table by date; for each month compute the percentage change against 12 months earlier and store it in a new column 'annual'. Do not download or invent other data. Print the row count and the number of missing values.

**Audit questions:** Was the table sorted? Is the result in percent? Was `periods=12` used? Were any rows lost? Are the first 12 months empty, and are there unexpected gaps after that?

**What does Check A test?** That the row count is unchanged; that in the sorted table the first 12 months are empty; that every computable month has a value and matches the correct calculation **exactly**; and that values are percent, not ratios. (If a month is missing on FRED, 12-month windows containing it are not compared.)

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(cpi)

## Task B — The peak month

From 2000-01 onwards, store the month with the **highest** annual inflation in `peak_month` (a date) and its value in `peak_value` (percent).

**Audit questions:** Is everything before 2000 excluded? Do the missing values (first 12 months) cause problems? Is the value in percent and a finite number (not NaN)?

**What does Check B test?** That `peak_month` is a date and the right month; that `peak_value` is a finite number matching the correct value.

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(peak_month, peak_value)

## Your audit notes

Before moving to the **audit log** in ML Lab, jot down briefly:
- What did you ask Gemini?
- What did you check in its code?
- What did you fix, or why did you change nothing?